## Эксперименты на этапе векторного поиска

#### Получение эмбедингов на основе разных моделей и сохранение их в pkl файл для дальнейшего локального сохранения в Qdrant и тестирования метрик

#### Получение эмбедингов (точек для Qdrant) с учетом чанков

In [4]:
import os
import json
import glob
import uuid
import pickle
import numpy as np
from typing import List, Dict, Any, Optional
from datetime import datetime
from dataclasses import dataclass, asdict
from tqdm.auto import tqdm
import torch
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

@dataclass
class ProcessedChunk:
    """Структура для сохраненного чанка"""
    chunk_id: str
    text: str
    original_text: str
    embedding: np.ndarray
    metadata: Dict[str, Any]
    document_id: str
    chunk_index: int
    total_chunks: int

class JsonPreparator:
    """
    Подготовка данных из JSON с сохранением в один файл.
    Использует tqdm для прогресс-баров и поддерживает GPU.
    """
    
    def __init__(
        self,
        json_path: str,
        embedding_model_name: str = "all-MiniLM-L6-v2",
        output_file: str = "./prepared_data.pkl",
        chunk_size: int = 1500,
        chunk_overlap: int = 150,
        batch_size: int = 100,
        use_gpu: bool = True,
        device: Optional[str] = None,

    ):
        """
        Args:
            json_path: Путь к JSON файлу или папке с JSON файлами
            embedding_model_name: Название модели эмбеддингов
            output_file: Путь к выходному файлу (.pkl или .json)
            chunk_size: Размер чанков в символах
            chunk_overlap: Перекрытие чанков
            batch_size: Размер батча для обработки
            use_gpu: Использовать GPU если доступен
            device: Конкретное устройство ('cuda:0', 'cpu', etc.)
        """
        self.json_path = json_path
        self.output_file = output_file
        self.batch_size = batch_size
        self.use_gpu = use_gpu
        
        if device:
            self.device = device
        elif use_gpu and torch.cuda.is_available():
            self.device = "cuda"
            print("🎮 GPU доступен и будет использован")
        else:
            self.device = "cpu"
            print("💻 Используется CPU")
        
        os.makedirs(os.path.dirname(output_file) if os.path.dirname(output_file) else '.', exist_ok=True)
        
        print(f"🚀 Загрузка модели эмбеддингов: {embedding_model_name}")
        print(f"   Устройство: {self.device}")
        
        self.emb_model = SentenceTransformer(embedding_model_name)

        self.emb_model.to(self.device)

        if self.device.startswith("cuda"):
            self._check_gpu_memory()
        
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=['\n\n', '\n', '. ', '! ', '? ', ' ', '']
        )

        self.all_chunks = []
        self.stats = {
            "processing_date": datetime.now().isoformat(),
            "embedding_model": embedding_model_name,
            "chunk_size": chunk_size,
            "chunk_overlap": chunk_overlap,
            "device": self.device,
            "total_documents": 0,
            "total_chunks": 0,
            "failed_documents": 0,
            "files_processed": 0,
            "total_tokens": 0,
            "total_chars": 0,
            "gpu_memory_used": 0,
            "processing_speed": 0
        }
    
    def _check_gpu_memory(self):
        """Проверка доступной памяти GPU"""
        try:
            if torch.cuda.is_available():
                gpu_name = torch.cuda.get_device_name(0)
                total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9  # в GB
                free_memory = torch.cuda.mem_get_info()[0] / 1e9  # в GB
                
                print(f"   GPU: {gpu_name}")
                print(f"   Память GPU: Всего {total_memory:.1f}GB, Свободно {free_memory:.1f}GB")
                
                if free_memory < 2:  # Меньше 2GB свободной памяти
                    print("   ⚠️  Внимание: Мало свободной памяти GPU!")
                    print("   Рекомендуется уменьшить batch_size или использовать CPU")
                
                return free_memory
        except Exception as e:
            print(f"   ⚠️  Не удалось проверить память GPU: {e}")
        
        return None
    
    def _load_json_data(self, filepath: str) -> List[Dict[str, Any]]:
        """Загрузка данных из JSON файла"""
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Обработка разных форматов JSON
        if isinstance(data, list):
            return data
        elif isinstance(data, dict):
            if 'documents' in data:
                return data['documents']
            elif 'articles' in data:
                return data['articles']
            else:
                return [data]
        return []
    
    def _format_chunk_text(self, chunk_text: str, document: Dict[str, Any]) -> str:
        """Форматирование текста чанка с добавлением заголовка"""
        title = document.get('title', '')
        if title:
            return f"Заголовок: {title}\n{chunk_text}"
        return chunk_text
    
    def _parse_timestamp(self, timestamp_str: str) -> Dict[str, Any]:
        """Парсинг временной метки"""        
        return {"raw": timestamp_str}
    
    def _create_metadata(self, document: Dict[str, Any], chunk_index: int, total_chunks: int) -> Dict[str, Any]:
        """Создание метаданных для чанка"""
        metadata = {
            "document_id": document.get('id', str(uuid.uuid4())),
            "title": document.get('title', ''),
            "chunk_index": chunk_index,
            "total_chunks": total_chunks,
            "original_length": len(document.get('text', '')),
            "chunk_length": 0,
            "processing_timestamp": datetime.now().isoformat()
        }
        
        time_field = document.get('published_time') or document.get('published-time')
        if time_field:
            metadata['published_time'] = self._parse_timestamp(str(time_field))
        
        # Добавляем все остальные поля
        for key, value in document.items():
            if key not in ['text', 'id', 'title', 'published_time', 'published-time']:
                if key not in metadata:
                    metadata[key] = value
        
        return metadata
    
    def process_documents_batch(self, documents: List[Dict[str, Any]]) -> List[ProcessedChunk]:
        """Обработка батча документов"""
        batch_chunks = []
        
        for doc in documents:
            if 'text' not in doc or not doc['text']:
                self.stats["failed_documents"] += 1
                continue
            
            doc_id = doc.get('id', str(uuid.uuid4()))
            text = doc['text']
            
            text_chunks = self.text_splitter.split_text(text)

            formatted_chunks = [self._format_chunk_text(chunk, doc) for chunk in text_chunks]
            
            try:
                if self.device.startswith("cuda"):
                    embeddings = self.emb_model.encode(
                        formatted_chunks,
                        show_progress_bar=False,
                        batch_size=min(len(formatted_chunks), 32),  # Оптимальный batch для GPU
                        convert_to_numpy=True,
                        device=self.device
                    )
                else:
                    embeddings = self.emb_model.encode(
                        formatted_chunks,
                        show_progress_bar=False
                    )

                if self.device.startswith("cuda"):
                    allocated = torch.cuda.memory_allocated() / 1e9
                    self.stats["gpu_memory_used"] = max(self.stats["gpu_memory_used"], allocated)
                    
            except RuntimeError as e:
                if "out of memory" in str(e):
                    print("   ⚠️  Нехватка памяти GPU! Пробуем уменьшить batch...")
                    embeddings = []
                    for chunk in formatted_chunks:
                        try:
                            emb = self.emb_model.encode(
                                [chunk],
                                show_progress_bar=False,
                                convert_to_numpy=True
                            )
                            embeddings.append(emb[0])
                        except:
                            embeddings.append(np.zeros(self.emb_model.get_sentence_embedding_dimension()))
                    embeddings = np.array(embeddings)
                else:
                    raise e
            
    
            for i, (chunk_text, formatted_text, embedding) in enumerate(zip(text_chunks, formatted_chunks, embeddings)):
                metadata = self._create_metadata(doc, i, len(text_chunks))
                metadata['chunk_length'] = len(chunk_text)
                
            
                if hasattr(embedding, 'cpu'):
                    embedding = embedding.cpu().numpy()
                elif hasattr(embedding, 'numpy'):
                    embedding = embedding.numpy()
                
                chunk = ProcessedChunk(
                    chunk_id=f"{doc_id}_chunk_{i}",
                    text=formatted_text,
                    original_text=chunk_text,
                    embedding=embedding.astype(np.float32),  
                    metadata=metadata,
                    document_id=doc_id,
                    chunk_index=i,
                    total_chunks=len(text_chunks)
                )
                batch_chunks.append(chunk)
                
      
                self.stats["total_chars"] += len(chunk_text)
                self.stats["total_tokens"] += len(chunk_text.split())
            
            self.stats["total_documents"] += 1
            self.stats["total_chunks"] += len(text_chunks)
        
        return batch_chunks
    
    def process_file(self, json_file: str):
        """Обработка одного JSON файла"""
        filename = os.path.basename(json_file)
        print(f"\n📄 Обработка файла: {filename}")
        
        try:
   
            documents = self._load_json_data(json_file)
            print(f"  📊 Загружено документов: {len(documents):,}")
            
   
            total_docs = len(documents)
            start_time = datetime.now()
            
            with tqdm(total=total_docs, desc=f"Обработка {filename[:20]}...", unit="док") as pbar:
                for i in range(0, total_docs, self.batch_size):
                    batch_docs = documents[i:i + self.batch_size]
                    
                    batch_chunks = self.process_documents_batch(batch_docs)
                    self.all_chunks.extend(batch_chunks)
                    
                    pbar.update(len(batch_docs))

                    elapsed = (datetime.now() - start_time).total_seconds()
                    docs_per_second = self.stats["total_documents"] / elapsed if elapsed > 0 else 0
                    
                    pbar.set_postfix({
                        'чанков': f"{len(self.all_chunks):,}",
                        'документов': f"{self.stats['total_documents']:,}",
                        'скорость': f"{docs_per_second:.1f} док/сек"
                    })
            
            total_time = (datetime.now() - start_time).total_seconds()
            self.stats["processing_speed"] = self.stats["total_documents"] / total_time if total_time > 0 else 0
            
            self.stats["files_processed"] += 1
            
            return {
                "file": json_file,
                "status": "success",
                "documents_processed": len(documents),
                "chunks_created": len(self.all_chunks),
                "processing_time_seconds": total_time
            }
            
        except Exception as e:
            print(f"❌ Ошибка при обработке файла {json_file}: {e}")
            return {
                "file": json_file,
                "status": "error",
                "error": str(e)
            }
    
    def process_all(self):
        """Обработка всех JSON файлов"""
        print("="*70)
        print("🚀 ЗАПУСК ПОДГОТОВКИ ДАННЫХ С GPU ПОДДЕРЖКОЙ")
        print("="*70)
        
        # Определяем файлы для обработки
        if os.path.isfile(self.json_path) and self.json_path.endswith('.json'):
            files = [self.json_path]
        elif os.path.isdir(self.json_path):
            files = glob.glob(os.path.join(self.json_path, '*.json'))
        else:
            raise ValueError(f"Неверный путь: {self.json_path}")
        
        print(f"📁 Найдено файлов для обработки: {len(files)}")
        print(f"💾 Выходной файл: {self.output_file}")
        print(f"🤖 Модель эмбеддингов: {self.stats['embedding_model']}")
        print(f"🎮 Устройство: {self.device}")
        print(f"✂️  Размер чанка: {self.stats['chunk_size']} символов")
        print(f"🔄 Перекрытие чанков: {self.stats['chunk_overlap']} символов")
        print(f"📦 Размер батча: {self.batch_size} документов")
        
        if self.device.startswith("cuda"):
            print(f"💿 Память GPU использовано: {self.stats['gpu_memory_used']:.2f} GB")
        
        print("="*70)
        
        if self.device.startswith("cuda"):
            torch.cuda.empty_cache()
        
        results = []
        total_start_time = datetime.now()
        
        for file in tqdm(files, desc="Обработка файлов", unit="файл"):
            result = self.process_file(file)
            results.append(result)
 
            if self.device.startswith("cuda") and len(results) % 5 == 0:
                torch.cuda.empty_cache()
        
        total_time = (datetime.now() - total_start_time).total_seconds()

        self._save_to_single_file()
        self._print_statistics(results)

        if self.device.startswith("cuda"):
            torch.cuda.empty_cache()
            print("🧹 Память GPU очищена")
        
        return results
    
    def _save_to_single_file(self):
        """Сохранение всех данных в один файл"""
        print(f"\n💾 Сохранение всех данных в файл: {self.output_file}")

        save_data = {
            'chunks': [],
            'embeddings': [],
            'metadata': [],
            'stats': self.stats
        }
        
        with tqdm(total=len(self.all_chunks), desc="Подготовка данных", unit="чанк") as pbar:
            for chunk in self.all_chunks:
                save_data['chunks'].append({
                    'chunk_id': chunk.chunk_id,
                    'text': chunk.text,
                    'original_text': chunk.original_text,
                    'document_id': chunk.document_id,
                    'chunk_index': chunk.chunk_index,
                    'total_chunks': chunk.total_chunks
                })
                save_data['embeddings'].append(chunk.embedding)
                save_data['metadata'].append(chunk.metadata)
                pbar.update(1)
        
        file_ext = os.path.splitext(self.output_file)[1].lower()
        
        if file_ext == '.pkl':
            with open(self.output_file, 'wb') as f:
                pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)
            print(f"✅ Данные сохранены в pickle файл: {self.output_file}")
            
        elif file_ext == '.npz':
            np.savez_compressed(
                self.output_file,
                embeddings=np.array(save_data['embeddings']),
                chunks=save_data['chunks'],
                metadata=save_data['metadata'],
                stats=save_data['stats']
            )
            print(f"✅ Данные сохранены в npz файл (со сжатием): {self.output_file}")
            
        elif file_ext == '.h5' or file_ext == '.hdf5':
            import h5py
            with h5py.File(self.output_file, 'w') as f:
                f.create_dataset('embeddings', data=np.array(save_data['embeddings']), compression='gzip')
                
                chunks_group = f.create_group('chunks')
                for i, chunk in enumerate(save_data['chunks']):
                    chunk_group = chunks_group.create_group(f'chunk_{i}')
                    for key, value in chunk.items():
                        if isinstance(value, str):
                            chunk_group.create_dataset(key, data=value)
                
                metadata_group = f.create_group('metadata')
                for i, meta in enumerate(save_data['metadata']):
                    meta_group = metadata_group.create_group(f'meta_{i}')
                    for key, value in meta.items():
                        if isinstance(value, (str, int, float)):
                            meta_group.create_dataset(key, data=value)
                        elif isinstance(value, dict):
                            # Для вложенных словарей
                            sub_group = meta_group.create_group(key)
                            for sub_key, sub_value in value.items():
                                if isinstance(sub_value, (str, int, float)):
                                    sub_group.create_dataset(sub_key, data=sub_value)
            
            print(f"✅ Данные сохранены в HDF5 файл: {self.output_file}")
            
        else:
            with open(self.output_file, 'wb') as f:
                pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)
            print(f"✅ Данные сохранены в pickle файл: {self.output_file}")
    
    def _print_statistics(self, results: List[Dict[str, Any]]):
        """Вывод статистики обработки"""
        successful = [r for r in results if r.get('status') == 'success']
        
        print("\n" + "="*70)
        print("📊 ИТОГОВАЯ СТАТИСТИКА")
        print("="*70)
        print(f"📁 Файлов обработано: {len(successful)}/{len(results)}")
        print(f"📄 Документов обработано: {self.stats['total_documents']:,}")
        print(f"✂️  Чанков создано: {self.stats['total_chunks']:,}")
        print(f"📝 Всего символов: {self.stats['total_chars']:,}")
        print(f"🔤 Примерное количество токенов: {self.stats['total_tokens']:,}")
        
        if self.stats['total_documents'] > 0:
            avg_chunks = self.stats['total_chunks'] / self.stats['total_documents']
            avg_chars = self.stats['total_chars'] / self.stats['total_chunks']
            print(f"📊 Среднее чанков на документ: {avg_chunks:.2f}")
            print(f"📊 Средний размер чанка: {avg_chars:.0f} символов")
        
        print("="*70)

# Тестирование метрик на разных моделях
#"Qwen/Qwen3-Embedding-0.6B",
#"Octen/Octen-Embedding-0.6B",
#"thenlper/gte-large",
#"BAAI/bge-small-en-v1.5"


CONFIG = {
        "json_path": "/kaggle/input/crunch/techcrunch_ai_5488_articles_20260112_1535.json",
        "embedding_model": "Qwen/Qwen3-Embedding-0.6B",
        "output_file": "qwen_prepare_data.pkl",
        "chunk_size": 500,
        "chunk_overlap": 50,
        "batch_size": 300
    }
use_gpu = torch.cuda.is_available()
device = "cuda" if use_gpu else "cpu"

preparator = JsonPreparator(
    json_path=CONFIG["json_path"],
    embedding_model_name=CONFIG["embedding_model"],
    output_file=CONFIG["output_file"],
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
    batch_size = CONFIG["batch_size"],
    use_gpu = use_gpu,
    device = device
)

results = preparator.process_all()


🚀 Загрузка модели эмбеддингов: Qwen/Qwen3-Embedding-0.6B
   Устройство: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...


📄 Обработка файла: techcrunch_ai_5488_articles_20260112_1535.json
  📊 Загружено документов: 5,791


Обработка techcrunch_ai_5488_a...:   0%|          | 0/5791 [00:00<?, ?док/s]


💾 Сохранение всех данных в файл: qwen_prepare_data.pkl


Подготовка данных:   0%|          | 0/70732 [00:00<?, ?чанк/s]

✅ Данные сохранены в pickle файл: qwen_prepare_data.pkl

📊 ИТОГОВАЯ СТАТИСТИКА
📁 Файлов обработано: 1/1
📄 Документов обработано: 5,791
✂️  Чанков создано: 70,732
📝 Всего символов: 24,833,926
🔤 Примерное количество токенов: 3,994,122
📊 Среднее чанков на документ: 12.21
📊 Средний размер чанка: 351 символов
🧹 Память GPU очищена


#### Получение эмбедингов без чанкинга

In [4]:
import os
import json
import glob
import uuid
import pickle
import numpy as np
from typing import List, Dict, Any, Optional
from datetime import datetime
from dataclasses import dataclass, asdict
from tqdm.auto import tqdm
import torch
from sentence_transformers import SentenceTransformer

@dataclass
class ProcessedChunk:
    """Структура для сохраненного чанка (каждый документ = один чанк)"""
    chunk_id: str
    text: str
    original_text: str
    embedding: np.ndarray
    metadata: Dict[str, Any]
    document_id: str
    chunk_index: int
    total_chunks: int

class JsonPreparator:
    """
    Подготовка данных из JSON с сохранением в формате chunks (каждый документ = 1 чанк).
    Использует tqdm для прогресс-баров и поддерживает GPU.
    """
    
    def __init__(
        self,
        json_path: str,
        embedding_model_name: str = "all-MiniLM-L6-v2",
        output_file: str = "./prepared_data.pkl",
        batch_size: int = 100,
        use_gpu: bool = True,
        device: Optional[str] = None,
    ):
        """
        Args:
            json_path: Путь к JSON файлу или папке с JSON файлами
            embedding_model_name: Название модели эмбеддингов
            output_file: Путь к выходному файлу (.pkl)
            batch_size: Размер батча для обработки
            use_gpu: Использовать GPU если доступен
            device: Конкретное устройство ('cuda:0', 'cpu', etc.)
        """
        self.json_path = json_path
        self.output_file = output_file
        self.batch_size = batch_size
        self.use_gpu = use_gpu
        
        # Определяем устройство
        if device:
            self.device = device
        elif use_gpu and torch.cuda.is_available():
            self.device = "cuda"
            print("🎮 GPU доступен и будет использован")
        else:
            self.device = "cpu"
            print("💻 Используется CPU")
        
        # Создаем директорию для выходного файла если нужно
        os.makedirs(os.path.dirname(output_file) if os.path.dirname(output_file) else '.', exist_ok=True)
        
        # Инициализация модели эмбеддингов с поддержкой GPU
        print(f"🚀 Загрузка модели эмбеддингов: {embedding_model_name}")
        print(f"   Устройство: {self.device}")
        
        self.emb_model = SentenceTransformer(embedding_model_name)
        
        # Перемещаем модель на нужное устройство
        self.emb_model.to(self.device)

        if self.device.startswith("cuda"):
            self._check_gpu_memory()
        
        # Данные для сохранения (каждый документ = один чанк)
        self.all_chunks = []
        self.stats = {
            "processing_date": datetime.now().isoformat(),
            "embedding_model": embedding_model_name,
            "device": self.device,
            "total_documents": 0,
            "total_chunks": 0,
            "failed_documents": 0,
            "files_processed": 0,
            "total_tokens": 0,
            "total_chars": 0,
            "gpu_memory_used": 0,
            "processing_speed": 0,
            "one_chunk_per_document": True  # Флаг, что каждый документ = 1 чанк
        }
    
    def _check_gpu_memory(self):
        """Проверка доступной памяти GPU"""
        try:
            if torch.cuda.is_available():
                gpu_name = torch.cuda.get_device_name(0)
                total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9  # в GB
                free_memory = torch.cuda.mem_get_info()[0] / 1e9  # в GB
                
                print(f"   GPU: {gpu_name}")
                print(f"   Память GPU: Всего {total_memory:.1f}GB, Свободно {free_memory:.1f}GB")
                
                if free_memory < 2:  # Меньше 2GB свободной памяти
                    print("   ⚠️  Внимание: Мало свободной памяти GPU!")
                    print("   Рекомендуется уменьшить batch_size или использовать CPU")
                
                return free_memory
        except Exception as e:
            print(f"   ⚠️  Не удалось проверить память GPU: {e}")
        
        return None
    
    def _load_json_data(self, filepath: str) -> List[Dict[str, Any]]:
        """Загрузка данных из JSON файла"""
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Обработка разных форматов JSON
        if isinstance(data, list):
            return data
        elif isinstance(data, dict):
            if 'documents' in data:
                return data['documents']
            elif 'articles' in data:
                return data['articles']
            else:
                return [data]
        return []
    
    def _format_text(self, text: str, document: Dict[str, Any]) -> str:
        """Форматирование текста с добавлением заголовка"""
        title = document.get('title', '')
        if title and not text.startswith(title):
            return f"Заголовок: {title}\n\n{text}"
        return text
    
    def _parse_timestamp(self, timestamp_str: str) -> Dict[str, Any]:
        """Парсинг временной метки"""        
        return {"raw": timestamp_str}
    
    def _create_metadata(self, document: Dict[str, Any], chunk_index: int, total_chunks: int) -> Dict[str, Any]:
        """Создание метаданных для чанка"""
        metadata = {
            "document_id": document.get('id', str(uuid.uuid4())),
            "title": document.get('title', ''),
            "chunk_index": chunk_index,
            "total_chunks": total_chunks,
            "original_length": len(document.get('text', '')),
            "chunk_length": len(document.get('text', '')),
            "processing_timestamp": datetime.now().isoformat(),
            "source_file": os.path.basename(self.json_path) if os.path.isfile(self.json_path) else "multiple_files",
            "is_full_document": True  # Флаг, что это полный документ (не разделенный на чанки)
        }
        
        # Добавляем время публикации
        time_field = document.get('published_time') or document.get('published-time')
        if time_field:
            metadata['published_time'] = self._parse_timestamp(str(time_field))
        
        # Добавляем все остальные поля
        for key, value in document.items():
            if key not in ['text', 'id', 'title', 'published_time', 'published-time']:
                if key not in metadata:
                    # Конвертируем несериализуемые типы
                    if isinstance(value, (np.integer, np.floating, np.bool_)):
                        metadata[key] = value.item()
                    elif isinstance(value, np.ndarray):
                        metadata[key] = value.tolist()
                    else:
                        metadata[key] = value
        
        return metadata
    
    def process_documents_batch(self, documents: List[Dict[str, Any]]) -> List[ProcessedChunk]:
        """Обработка батча документов (каждый документ = один чанк)"""
        batch_chunks = []
        
        # Собираем тексты для батчевой обработки
        texts_to_encode = []
        valid_docs = []
        
        for doc in documents:
            if 'text' not in doc or not doc['text']:
                self.stats["failed_documents"] += 1
                continue
            
            text = doc['text']
            formatted_text = self._format_text(text, doc)
            
            texts_to_encode.append(formatted_text)
            valid_docs.append(doc)
        
        if not texts_to_encode:
            return []
        
        # Создаем эмбеддинги для всех документов батча на GPU
        try:
            if self.device.startswith("cuda"):
                embeddings = self.emb_model.encode(
                    texts_to_encode,
                    show_progress_bar=False,
                    batch_size=min(len(texts_to_encode), 32),  # Оптимальный batch для GPU
                    convert_to_numpy=True,
                    device=self.device
                )
                
                # Проверяем использование памяти GPU
                allocated = torch.cuda.memory_allocated() / 1e9
                self.stats["gpu_memory_used"] = max(self.stats["gpu_memory_used"], allocated)
            else:
                embeddings = self.emb_model.encode(
                    texts_to_encode,
                    show_progress_bar=False
                )
                
        except RuntimeError as e:
            if "out of memory" in str(e):
                print("   ⚠️  Нехватка памяти GPU! Пробуем обработать по одному...")
                # Пробуем обработать по одному
                embeddings = []
                for text in texts_to_encode:
                    try:
                        emb = self.emb_model.encode(
                            [text],
                            show_progress_bar=False,
                            convert_to_numpy=True,
                            device=self.device
                        )
                        embeddings.append(emb[0])
                    except:
                        # Если все равно ошибка, создаем нулевой эмбеддинг
                        print("   ❌ Ошибка при обработке документа, пропускаем")
                        embeddings.append(np.zeros(self.emb_model.get_sentence_embedding_dimension()))
                        self.stats["failed_documents"] += 1
                embeddings = np.array(embeddings)
            else:
                raise e
        
        # Создаем объекты ProcessedChunk (каждый документ = один чанк)
        for i, (doc, formatted_text, embedding) in enumerate(zip(valid_docs, texts_to_encode, embeddings)):
            doc_id = doc.get('id', str(uuid.uuid4()))
            original_text = doc['text']

            metadata = self._create_metadata(doc, chunk_index=0, total_chunks=1)
            
            if hasattr(embedding, 'cpu'):
                embedding = embedding.cpu().numpy()
            elif hasattr(embedding, 'numpy'):
                embedding = embedding.numpy()
            
            chunk = ProcessedChunk(
                chunk_id=f"{doc_id}_full",
                text=formatted_text,
                original_text=original_text,
                embedding=embedding.astype(np.float32),  # Всегда сохраняем как float32
                metadata=metadata,
                document_id=doc_id,
                chunk_index=0,  
                total_chunks=1   
            )
            batch_chunks.append(chunk)
            
            self.stats["total_chars"] += len(original_text)
            self.stats["total_tokens"] += len(original_text.split())
            self.stats["total_documents"] += 1
            self.stats["total_chunks"] += 1
        
        return batch_chunks
    
    def process_file(self, json_file: str):
        """Обработка одного JSON файла"""
        filename = os.path.basename(json_file)
        print(f"\n📄 Обработка файла: {filename}")
        
        try:
            documents = self._load_json_data(json_file)
            print(f"  📊 Загружено документов: {len(documents):,}")
            

            total_docs = len(documents)
            start_time = datetime.now()
            
            with tqdm(total=total_docs, desc=f"Обработка {filename[:20]}...", unit="док") as pbar:
                for i in range(0, total_docs, self.batch_size):
                    batch_docs = documents[i:i + self.batch_size]
                    
                    batch_chunks = self.process_documents_batch(batch_docs)
                    self.all_chunks.extend(batch_chunks)
                    
                    pbar.update(len(batch_docs))

                    elapsed = (datetime.now() - start_time).total_seconds()
                    docs_per_second = self.stats["total_documents"] / elapsed if elapsed > 0 else 0
                    
                    pbar.set_postfix({
                        'чанков': f"{len(self.all_chunks):,}",
                        'скорость': f"{docs_per_second:.1f} док/сек"
                    })

            total_time = (datetime.now() - start_time).total_seconds()
            self.stats["processing_speed"] = self.stats["total_documents"] / total_time if total_time > 0 else 0
            
            self.stats["files_processed"] += 1
            
            return {
                "file": json_file,
                "status": "success",
                "documents_processed": len(documents),
                "chunks_created": len(self.all_chunks),
                "processing_time_seconds": total_time
            }
            
        except Exception as e:
            print(f"❌ Ошибка при обработке файла {json_file}: {e}")
            import traceback
            traceback.print_exc()
            return {
                "file": json_file,
                "status": "error",
                "error": str(e)
            }
    
    def process_all(self):
        """Обработка всех JSON файлов"""
        print("="*70)
        print("🚀 ЗАПУСК ПОДГОТОВКИ ДАННЫХ (ОДИН ЧАНК НА ДОКУМЕНТ)")
        print("="*70)
        
        if os.path.isfile(self.json_path) and self.json_path.endswith('.json'):
            files = [self.json_path]
        elif os.path.isdir(self.json_path):
            files = glob.glob(os.path.join(self.json_path, '*.json'))
        else:
            raise ValueError(f"Неверный путь: {self.json_path}")
        
        print(f"📁 Найдено файлов для обработки: {len(files)}")
        print(f"💾 Выходной файл: {self.output_file}")
        print(f"🤖 Модель эмбеддингов: {self.stats['embedding_model']}")
        print(f"🎮 Устройство: {self.device}")
        print(f"📦 Размер батча: {self.batch_size} документов")
        print(f"📄 Режим: ОДИН ЧАНК НА ДОКУМЕНТ (полные тексты)")
        
        if self.device.startswith("cuda"):
            print(f"💿 Память GPU использовано: {self.stats['gpu_memory_used']:.2f} GB")
        
        print("="*70)

        if self.device.startswith("cuda"):
            torch.cuda.empty_cache()
        
        results = []
        total_start_time = datetime.now()
        
        for file in tqdm(files, desc="Обработка файлов", unit="файл"):
            result = self.process_file(file)
            results.append(result)

            if self.device.startswith("cuda") and len(results) % 5 == 0:
                torch.cuda.empty_cache()
        
        total_time = (datetime.now() - total_start_time).total_seconds()
        

        self._save_to_single_file()
        self._print_statistics(results, total_time)
        
        if self.device.startswith("cuda"):
            torch.cuda.empty_cache()
            print("🧹 Память GPU очищена")
        
        return results
    
    def _save_to_single_file(self):
        """Сохранение всех данных в один файл в формате chunks"""
        print(f"\n💾 Сохранение всех данных в файл: {self.output_file}")
        
        save_data = {
            'chunks': [],      # Список чанков (каждый документ = 1 чанк)
            'embeddings': [],  # Эмбеддинги для каждого чанка
            'metadata': [],    # Метаданные для каждого чанка
            'stats': self.stats
        }

        with tqdm(total=len(self.all_chunks), desc="Подготовка данных", unit="чанк") as pbar:
            for chunk in self.all_chunks:
                chunk_info = {
                    'chunk_id': chunk.chunk_id,
                    'text': chunk.text,              
                    'original_text': chunk.original_text,  
                    'document_id': chunk.document_id,
                    'chunk_index': chunk.chunk_index,
                    'total_chunks': chunk.total_chunks
                }
                save_data['chunks'].append(chunk_info)
                
                save_data['embeddings'].append(chunk.embedding)
                save_data['metadata'].append(chunk.metadata)
                
                pbar.update(1)
        

        with open(self.output_file, 'wb') as f:
            pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)
            
        info_file = self.output_file.replace('.pkl', '_info.json')
        with open(info_file, 'w', encoding='utf-8') as f:
            json.dump({
                'total_documents': len(self.all_chunks),
                'total_chunks': len(self.all_chunks),
                'embedding_dim': self.all_chunks[0].embedding.shape if self.all_chunks else 0,
                'stats': self.stats,
                'format_description': 'Каждый документ сохранен как один чанк (полный текст без разбиения).',
                'structure': {
                    'chunks': 'список чанков (каждый документ = 1 чанк)',
                    'embeddings': 'эмбеддинги для каждого чанка',
                    'metadata': 'метаданные для каждого чанка'
                }
            }, f, ensure_ascii=False, indent=2)
        
        print(f"✅ Данные сохранены в pickle файл: {self.output_file}")
        print(f"✅ Информация о файле сохранена в: {info_file}")
        print(f"📊 Всего документов/чанков: {len(self.all_chunks):,}")
        print(f"📊 Размерность эмбеддингов: {self.all_chunks[0].embedding.shape if self.all_chunks else 'N/A'}")
        print(f"📊 Формат данных: chunks (каждый документ = 1 чанк)")
    
    def _print_statistics(self, results: List[Dict[str, Any]], total_time: float):
        """Вывод статистики обработки"""
        successful = [r for r in results if r.get('status') == 'success']
        
        print("\n" + "="*70)
        print("📊 ИТОГОВАЯ СТАТИСТИКА (ОДИН ЧАНК НА ДОКУМЕНТ)")
        print("="*70)
        print(f"📁 Файлов обработано: {len(successful)}/{len(results)}")
        print(f"📄 Документов обработано: {self.stats['total_documents']:,}")
        print(f"✂️  Чанков создано: {self.stats['total_chunks']:,} (каждый документ = 1 чанк)")
        print(f"❌ Документов с ошибками: {self.stats['failed_documents']:,}")
        print(f"📝 Всего символов: {self.stats['total_chars']:,}")
        print(f"🔤 Примерное количество токенов: {self.stats['total_tokens']:,}")
        
        if self.stats['total_documents'] > 0:
            avg_chars = self.stats['total_chars'] / self.stats['total_documents']
            print(f"📊 Средний размер документа: {avg_chars:.0f} символов")
            print(f"📊 Среднее количество токенов на документ: {self.stats['total_tokens'] / self.stats['total_documents']:.0f}")
        
        print(f"⏱️  Общее время обработки: {total_time:.2f} секунд")
        print(f"⚡ Средняя скорость: {self.stats['total_documents'] / total_time:.2f} документов/сек")
        
        if self.device.startswith("cuda"):
            print(f"💿 Максимальное использование памяти GPU: {self.stats['gpu_memory_used']:.2f} GB")
        
        print("="*70)
        
        if self.all_chunks:
            print("\n🔍 ПРИМЕР ПЕРВОГО ЧАНКА:")
            print(f"Chunk ID: {self.all_chunks[0].chunk_id}")
            print(f"Document ID: {self.all_chunks[0].document_id}")
            print(f"Заголовок: {self.all_chunks[0].metadata.get('title', 'N/A')}")
            print(f"Chunk index: {self.all_chunks[0].chunk_index}/{self.all_chunks[0].total_chunks}")
            print(f"Длина текста: {len(self.all_chunks[0].original_text)} символов")
            print(f"Текст (первые 300 символов):")
            print(self.all_chunks[0].original_text[:300] + "...")
            print("="*70)

CONFIG = {
    "json_path": "/kaggle/input/crunch/techcrunch_ai_5488_articles_20260112_1535.json",
    "embedding_model": "BAAI/bge-small-en-v1.5",
    "output_file": "bge_prepare_data_no_chunks.pkl",  # Файл с чанками (1 документ = 1 чанк)
    "batch_size": 300
}

use_gpu = torch.cuda.is_available()
device = "cuda" if use_gpu else "cpu"

preparator = JsonPreparator(
    json_path=CONFIG["json_path"],
    embedding_model_name=CONFIG["embedding_model"],
    output_file=CONFIG["output_file"],
    batch_size=CONFIG["batch_size"],
    use_gpu=use_gpu,
    device=device
)

results = preparator.process_all()

🚀 Загрузка модели эмбеддингов: BAAI/bge-small-en-v1.5
   Устройство: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...


📄 Обработка файла: techcrunch_ai_5488_articles_20260112_1535.json
  📊 Загружено документов: 5,791


Обработка techcrunch_ai_5488_a...:   0%|          | 0/5791 [00:00<?, ?док/s]


💾 Сохранение всех данных в файл: bge_prepare_data_no_chunks.pkl


Подготовка данных:   0%|          | 0/5791 [00:00<?, ?чанк/s]

✅ Данные сохранены в pickle файл: bge_prepare_data_no_chunks.pkl
✅ Информация о файле сохранена в: bge_prepare_data_no_chunks_info.json
📊 Всего документов/чанков: 5,791
📊 Размерность эмбеддингов: (384,)
📊 Формат данных: chunks (каждый документ = 1 чанк)

📊 ИТОГОВАЯ СТАТИСТИКА (ОДИН ЧАНК НА ДОКУМЕНТ)
📁 Файлов обработано: 1/1
📄 Документов обработано: 5,791
✂️  Чанков создано: 5,791 (каждый документ = 1 чанк)
❌ Документов с ошибками: 0
📝 Всего символов: 24,897,565
🔤 Примерное количество токенов: 3,980,896
📊 Средний размер документа: 4299 символов
📊 Среднее количество токенов на документ: 687
⏱️  Общее время обработки: 40.32 секунд
⚡ Средняя скорость: 143.62 документов/сек
💿 Максимальное использование памяти GPU: 6.25 GB

🔍 ПРИМЕР ПЕРВОГО ЧАНКА:
Chunk ID: bd09eee1-4882-5133-ada4-adc73a1c38e4_full
Document ID: bd09eee1-4882-5133-ada4-adc73a1c38e4
Заголовок: Sam Altman would like remind you that humans use a lot of energy, too
Chunk index: 0/1
Длина текста: 3070 символов
Текст (первые 300 си

### Получение разных эмбедингов вопросов

In [1]:
import json
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from datetime import datetime

def prepare_embeddings(
    input_json: str,
    output_json: str,
    models: list,
    batch_size: int = 32
):
    """
    Создает JSON с эмбеддингами для вопросов по разным моделям
    """
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"💻 Устройство: {device}")
    
    print(f"📂 Загрузка {input_json}...")
    with open(input_json, 'r', encoding='utf-8') as f:
        questions_data = json.load(f)
    
    print(f"✅ Загружено {len(questions_data)} вопросов")
    
    for model_name in models:
        print(f"\n🔄 Модель: {model_name}")
        
        model = SentenceTransformer(model_name, device=device)
        print(f"   📊 Размерность: {model.get_sentence_embedding_dimension()}")
        
        texts = [item['question'] for item in questions_data]
        
        embeddings = []
        for i in tqdm(range(0, len(texts), batch_size), desc="   Прогресс"):
            batch = texts[i:i+batch_size]
            with torch.no_grad():
                batch_emb = model.encode(batch, convert_to_numpy=True)
                embeddings.extend(batch_emb.tolist())
        
        model_key = model_name.replace('/', '_').replace('-', '_')
        for i, item in enumerate(questions_data):
            if 'embeddings' not in item:
                item['embeddings'] = {}
            item['embeddings'][model_key] = embeddings[i]

        del model
        if device == 'cuda':
            torch.cuda.empty_cache()

    print(f"\n💾 Сохранение в {output_json}...")
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(questions_data, f, ensure_ascii=False, indent=2)
    
    print("✅ Готово!")

if __name__ == "__main__":
    models = [
        "Qwen/Qwen3-Embedding-0.6B",
        "Octen/Octen-Embedding-0.6B",
        "thenlper/gte-large",
        "BAAI/bge-small-en-v1.5"
    ]
    
    prepare_embeddings(
        input_json="/kaggle/input/crunch/questions.json",
        output_json="questions_with_embeds.json",
        models=models,
        batch_size=32
    )

2026-03-01 13:14:30.152897: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772370870.461762      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772370870.562190      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772370871.352058      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772370871.352104      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772370871.352110      55 computation_placer.cc:177] computation placer alr

💻 Устройство: cuda
📂 Загрузка /kaggle/input/crunch/questions.json...
✅ Загружено 80 вопросов

🔄 Модель: Qwen/Qwen3-Embedding-0.6B


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

   📊 Размерность: 384


   Прогресс: 100%|██████████| 3/3 [00:00<00:00, 62.04it/s]


💾 Сохранение в questions_with_embeds.json...


✅ Готово!
